
# Example 2 — Nonlinear cylinder target in \(d=2\): one chunk splits into two

This notebook instantiates the **collective nonlinear cylinder terminal condition** from Section 3.3 / **Algorithm 2** and follows the paper's suggested **two-region mass-splitting** example.

The setup is:
- start from **one compact chunk** of mass in the middle
- define two smooth periodic bump observables centered in two disjoint upper regions
- push both observable values upward at terminal time, so the mass is encouraged to **split into two lobes**

As in the first notebook, particle mass is shown by color and the main output is an interactive slider.


In [ ]:
import numpy as np
import sys
from pathlib import Path


ROOT_CANDIDATES = (Path.cwd().resolve(), *Path.cwd().resolve().parents, Path("/mnt/data").resolve())
ROOT = next(
    (candidate for candidate in ROOT_CANDIDATES if (candidate / "wasserstein_conditioning_algorithms.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate wasserstein_conditioning_algorithms.py")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.wasserstein_conditioning_algorithms import simulate_nonlinear_cylinder_quadrature_em

np.set_printoptions(precision=3, suppress=True)


In [ ]:
import plotly.graph_objects as go

from notebooks.support import (
    center_trace,
    circle_trace,
    configure_plotly,
    line_trace as _line_trace,
    make_particle_animation as _make_particle_animation,
)

configure_plotly()


def line_trace(points, name, color="rgba(80,80,80,0.55)", dash="dot", close=False, showlegend=True, marker_size=6):
    return _line_trace(
        points,
        name,
        color=color,
        dash=dash,
        close=close,
        showlegend=showlegend,
        marker_size=marker_size,
        mode="lines+markers",
    )



def make_particle_animation(
    positions,
    times,
    masses,
    title,
    static_traces=None,
    marker_size=18,
    x_range=(0.0, 1.0),
    y_range=(0.0, 1.0),
):
    return _make_particle_animation(
        positions,
        times,
        masses,
        title,
        static_traces=static_traces,
        marker_size=marker_size,
        x_range=x_range,
        y_range=y_range,
        mass_format=".3f",
        time_formatter=lambda t, h: f"time = {t:.3f}",
        slider_label_formatter=lambda t, h: f"{t:.3f}",
        currentvalue_prefix="time = ",
        width=800,
        height=700,
        play_frame_duration=110,
    )


In [ ]:

def periodic_bump(center, sharpness=1.0):
    center = np.asarray(center, dtype=float)

    def phi(x):
        x = np.asarray(x, dtype=float)
        dx = x[..., 0] - center[0]
        dy = x[..., 1] - center[1]
        return np.exp(
            sharpness * (
                np.cos(2.0 * np.pi * dx)
                + np.cos(2.0 * np.pi * dy)
                - 2.0
            )
        )

    return phi

def gauss_hermite_gaussian_quadrature(lambda_, order_per_dim=5, dim=2):
    '''
    Build a tensor-product Gauss-Hermite rule for
        (2πλ)^(-K/2) exp(-|η|^2/(2λ)) dη.
    '''
    from numpy.polynomial.hermite import hermgauss

    nodes_1d, weights_1d = hermgauss(order_per_dim)

    node_mesh = np.meshgrid(*([nodes_1d] * dim), indexing="ij")
    weight_mesh = np.meshgrid(*([weights_1d] * dim), indexing="ij")

    xi = np.stack([m.ravel() for m in node_mesh], axis=-1)
    w = np.prod(np.stack(weight_mesh, axis=-1), axis=-1)

    eta = np.sqrt(2.0 * lambda_) * xi
    weights = w / (np.pi ** (dim / 2.0))
    return eta, weights

# A small number of particles keeps the masses reasonably large and produces
# a clear visual split.
masses = np.array([0.26, 0.20, 0.17, 0.14, 0.13, 0.10], dtype=float)

# Start from one compact chunk near the middle.
initial_positions = np.array([
    [0.50, 0.46],
    [0.52, 0.50],
    [0.48, 0.54],
    [0.54, 0.46],
    [0.46, 0.50],
    [0.50, 0.54],
], dtype=float)

# Two target regions in the upper-left and upper-right.
region_centers = np.array([
    [0.30, 0.70],
    [0.70, 0.70],
], dtype=float)

sharpness = 1.0
observables = [periodic_bump(center, sharpness=sharpness) for center in region_centers]

# Ask both observables to become moderately large, which encourages a split.
target_vector = np.array([0.35, 0.35], dtype=float)

lambda_ = 10000
horizon = 0.08
step_size = 0.004
grid_shape = 24
quadrature_order = 5
seed = 28

quadrature_nodes, quadrature_weights = gauss_hermite_gaussian_quadrature(
    lambda_=lambda_,
    order_per_dim=quadrature_order,
    dim=2,
)

print("number of particles:", len(masses))
print("region centers:\n", region_centers)
print("target vector:", target_vector)
print("grid_shape:", grid_shape, "quadrature nodes:", len(quadrature_weights))


In [ ]:

rng = np.random.default_rng(seed)

sim = simulate_nonlinear_cylinder_quadrature_em(
    masses=masses,
    observables=observables,
    target_vector=target_vector,
    lambda_=lambda_,
    horizon=horizon,
    step_size=step_size,
    initial_positions=initial_positions,
    quadrature_nodes=quadrature_nodes,
    quadrature_weights=quadrature_weights,
    grid_shape=grid_shape,
    rng=rng,
    store_drifts=True,
)

print("positions array shape:", sim.positions.shape)
print("final time:", float(sim.times[-1]))


In [ ]:

static_traces = [
    circle_trace(region_centers[0], radius=0.14, name="left target region", color="rgba(65, 105, 225, 0.55)"),
    circle_trace(region_centers[1], radius=0.14, name="right target region", color="rgba(220, 20, 60, 0.55)"),
    center_trace(region_centers[[0]], name="left center", color="rgba(65, 105, 225, 0.9)", symbol="x", size=12, showlegend=False),
    center_trace(region_centers[[1]], name="right center", color="rgba(220, 20, 60, 0.9)", symbol="x", size=12, showlegend=False),
]

fig = make_particle_animation(
    positions=sim.positions,
    times=sim.times,
    masses=sim.masses,
    title="Example 2: Nonlinear cylinder target (one chunk splits into two)",
    static_traces=static_traces,
    marker_size=22,
)
fig.show()


In [ ]:

# Track the two collective moments μ_t(φ_1) and μ_t(φ_2).
moment_values = np.array([
    [np.dot(sim.masses, phi(pos)) for phi in observables]
    for pos in sim.positions
])

import plotly.graph_objects as go

moment_fig = go.Figure()
moment_fig.add_trace(go.Scatter(x=sim.times, y=moment_values[:, 0], mode="lines+markers", name="μ_t(φ₁)"))
moment_fig.add_trace(go.Scatter(x=sim.times, y=moment_values[:, 1], mode="lines+markers", name="μ_t(φ₂)"))

moment_fig.add_hline(y=float(target_vector[0]), line_dash="dash", line_color="rgba(65, 105, 225, 0.7)")
moment_fig.add_hline(y=float(target_vector[1]), line_dash="dash", line_color="rgba(220, 20, 60, 0.7)")

moment_fig.update_layout(
    title="Collective observable values over time",
    template="simple_white",
    width=760,
    height=440,
    xaxis_title="time",
    yaxis_title="observable value",
)
moment_fig.show()

print("initial moments:", moment_values[0])
print("final moments:", moment_values[-1])


In [ ]:

# A simple coarse left/right mass diagnostic.
left_mass = np.array([
    sim.masses[pos[:, 0] < 0.5].sum()
    for pos in sim.positions
])
right_mass = 1.0 - left_mass

split_fig = go.Figure()
split_fig.add_trace(go.Scatter(x=sim.times, y=left_mass, mode="lines+markers", name="mass with x < 0.5"))
split_fig.add_trace(go.Scatter(x=sim.times, y=right_mass, mode="lines+markers", name="mass with x ≥ 0.5"))

split_fig.update_layout(
    title="Coarse left/right split over time",
    template="simple_white",
    width=760,
    height=420,
    xaxis_title="time",
    yaxis_title="total mass",
)
split_fig.show()

print("final left/right mass split:", float(left_mass[-1]), float(right_mass[-1]))
